In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("src","")
src_value=dbutils.widgets.get("src")
print(src_value)

In [0]:
if src_value=='payment':
    key_cols="['order_id','payment_sequential']"
else:
    key_cols=f"['{src_value}_id']"

key_cols_list=eval(key_cols)

cdc_col="modified_date"

backdated_col=""

source_schema="silver_olist"

source_object=f"silver_{src_value}s"

target_schema="gold_olist"

target_object=f"Dim{src_value}s"

surrogate_key=f"Dim{src_value}Key"

### **Incremental Data Ingestion**

In [0]:
if len(backdated_col)==0:
    if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
        last_load=spark.sql(f"select max({cdc_col}) from {target_schema}.{target_object}").collect()[0][0]
    else:
        last_load='1900-01-01 00:00:00'
else:
    last_load=backdated_col
print(last_load)

In [0]:
df_source=spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col} >'{last_load}'")

In [0]:
df_source.show()

In [0]:
key_cols_incremental=",".join([f" '' as {i}" for i in key_cols_list])
print(key_cols_incremental)

In [0]:
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    key_cols_init=",".join(key_cols_list)
    df_target=spark.sql(f"select {key_cols_init},{surrogate_key},updated_at,created_at from {target_schema}.{target_object}")
else:
    key_cols_incremental=",".join([f" '' as {i}" for i in key_cols_list])
    df_target=spark.sql(f"select {key_cols_incremental},cast('0' as long) as {surrogate_key},cast('1900-01-01 00:00:00' as timestamp) as updated_at,cast('1900-01-01 00:00:00' as timestamp) as created_at where 1=0")


In [0]:
join_condition=' AND '.join(f"src.{i}==trg.{i}"for i in key_cols_list)
print(join_condition)

In [0]:
df_source.createOrReplaceTempView("src")
df_target.createOrReplaceTempView("trg")
df_join=spark.sql(f"""
                  select 
                  src.*,
                  trg.{surrogate_key},
                  trg.updated_at,
                  trg.created_at
                  from src left join trg
                  on {join_condition}
                  """)

In [0]:
df_new=df_join.filter(col(f"{surrogate_key}").isNull())

In [0]:
df_old=df_join.filter(col(f"{surrogate_key}").isNotNull())
df_old_enriched=df_old.withColumn("updated_at",current_timestamp())

In [0]:
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    max_surrogate_key=spark.sql(f"select max({surrogate_key}) from {target_schema}.{target_object}").collect()[0][0]
    df_new_enriched=df_new.withColumn(f"{surrogate_key}",lit(0)+max_surrogate_key+monotonically_increasing_id())\
        .withColumn("updated_at",current_timestamp())
else:
    max_surrogate_key=0
    df_new_enriched=df_new.withColumn("updated_at",current_timestamp())\
        .withColumn("created_at",current_timestamp())\
        .withColumn(f"{surrogate_key}",lit(0)+max_surrogate_key+monotonically_increasing_id())

In [0]:
df_union=df_old_enriched.unionByName(df_new_enriched)

In [0]:
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    dlt_obj=DeltaTable.forName(spark,f"{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_union.alias("src"),f"src.{surrogate_key}=trg.{surrogate_key}").whenMatchedUpdateAll(condition=f"src.{cdc_col}>trg.{cdc_col}").whenNotMatchedInsertAll()
else:
    df_union.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.{target_object}")